In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

# 1. PROJECT PATHS


PROJECT_ROOT = Path(
    r"D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations"
)

FEATURE_DATA_DIR = PROJECT_ROOT / "data" / "feature_data"
ANALYSIS_DATA_DIR = PROJECT_ROOT / "data" / "analysis_data"

DATA_FILE = (
    FEATURE_DATA_DIR / "APL_Day3_Feature_Engineered.csv"
)

MASTER_KPI_FILE = (
    ANALYSIS_DATA_DIR / "APL_Day15_Master_KPIs.csv"
)

DASHBOARD_INVENTORY_FILE = (
    ANALYSIS_DATA_DIR / "APL_Day15_Dashboard_Data_Inventory.csv"
)

# 2. OUTPUT DIRECTORY


ANALYSIS_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project Root:", PROJECT_ROOT)
print("Primary Dataset:", DATA_FILE)
print("Analysis Output Directory:", ANALYSIS_DATA_DIR)

assert PROJECT_ROOT.exists(), "Project root does not exist."
assert DATA_FILE.exists(), "Day 3 feature-engineered dataset is missing."

print("\nDay 19 project paths validated.")

Project Root: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations
Primary Dataset: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\data\feature_data\APL_Day3_Feature_Engineered.csv
Analysis Output Directory: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\data\analysis_data

Day 19 project paths validated.


In [3]:
df = pd.read_csv(
    DATA_FILE,
    encoding="cp1252",
    low_memory=False
)

# Standardize column-name whitespace without changing the source file.

df.columns = df.columns.astype(str).str.strip()

print("Dataset Shape:", df.shape)

print("\nFirst 5 Records:")
display(df.head())

print("\nColumn Names:")
print(df.columns.tolist())

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nTotal Missing Cells:", df.isna().sum().sum())

print("\nData Types:")
display(df.dtypes.to_frame("Data_Type"))

Dataset Shape: (180519, 54)

First 5 Records:


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Delay_Severity,Delivery_Efficiency_Ratio,Revenue_Per_Unit,Profit_Margin,Discount_Level,Region_Market,Shipping_Mode_Rank,Is_Corporate_Customer,Order_Quantity_Category,Operational_Risk_Profile
0,DEBIT,6,4,159.69,472.45,Late delivery,1,9,Cardio Equipment,Brownsville,...,Major Delay,0.666667,99.99,0.319412,Low Discount,Pacific Asia - South Asia,4,0,Large,High Risk - Slower Shipping
1,DEBIT,4,4,48.71,167.96,Shipping on time,0,29,Shop By Sport,Littleton,...,On Time / Early,1.000000,39.99,0.243611,Medium Discount,LATAM - Central America,4,0,Large,Lower Risk
2,DEBIT,4,4,87.36,181.99,Shipping on time,0,48,Water Sports,Littleton,...,On Time / Early,1.000000,199.99,0.436822,Low Discount,LATAM - Central America,4,0,Small,Lower Risk
3,DEBIT,6,4,-41.89,175.99,Late delivery,1,48,Water Sports,Littleton,...,Major Delay,0.666667,199.99,-0.209460,Medium Discount,USCA - East of USA,4,0,Small,High Risk - Slower Shipping
4,DEBIT,6,4,10.00,40.00,Late delivery,1,24,Women's Apparel,Littleton,...,Major Delay,0.666667,50.00,0.200000,Medium Discount,USCA - East of USA,4,0,Small,High Risk - Slower Shipping



Column Names:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Product Name', 'Product Price', 'Shipping Mode', 'Shipping_Delay_Days', 'Is_Delayed', 'Is_On_Time', 'Early_Delivery_Days', 'Delay_Severity', 'Delivery_Efficiency_Ratio', 'Revenue_Per_Unit', 'Profit_Margin', 'Discount_Level', 'Region_Market', 'Shipping_Mode_Rank', 'Is_Corporate_C

,Data_Type
Type,str
Days for shipping (real),int64
Days for shipment (scheduled),int64
Benefit per order,float64
Sales per customer,float64
Delivery Status,str
Late_delivery_risk,int64
Category Id,int64
Category Name,str
Customer City,str


## Define the Diagnostic Variables

In [4]:
diagnostic_df = df.copy()

required_columns = [
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Delivery Status",
    "Late_delivery_risk",
    "Shipping Mode",
    "Order Region",
    "Market"
]

missing_required_columns = [
    col for col in required_columns
    if col not in diagnostic_df.columns
]

if missing_required_columns:
    raise ValueError(
        f"Required columns are missing: {missing_required_columns}"
    )

# Convert shipping-duration fields to numeric.

diagnostic_df["Actual_Shipping_Days"] = pd.to_numeric(
    diagnostic_df["Days for shipping (real)"],
    errors="coerce"
)

diagnostic_df["Scheduled_Shipping_Days"] = pd.to_numeric(
    diagnostic_df["Days for shipment (scheduled)"],
    errors="coerce"
)

# Calculate the difference between actual and scheduled duration.

diagnostic_df["Diagnostic_Delay_Days"] = (
    diagnostic_df["Actual_Shipping_Days"]
    - diagnostic_df["Scheduled_Shipping_Days"]
)

# Identify records with valid, non-negative durations.

valid_duration = (
    diagnostic_df["Actual_Shipping_Days"].notna()
    & diagnostic_df["Scheduled_Shipping_Days"].notna()
    & diagnostic_df["Actual_Shipping_Days"].ge(0)
    & diagnostic_df["Scheduled_Shipping_Days"].ge(0)
)

diagnostic_df["Valid_Duration_Record"] = valid_duration

# Classify actual delivery performance.

diagnostic_df["Diagnostic_Delivery_Outcome"] = np.select(
    [
        valid_duration
        & diagnostic_df["Diagnostic_Delay_Days"].gt(0),

        valid_duration
        & diagnostic_df["Diagnostic_Delay_Days"].le(0)
    ],
    [
        "Delayed",
        "On Schedule or Early"
    ],
    default="Unknown"
)

print("Diagnostic dataset shape:", diagnostic_df.shape)

print("\nValid duration records:")
print(diagnostic_df["Valid_Duration_Record"].sum())

print("\nDelivery outcome distribution:")
display(
    diagnostic_df["Diagnostic_Delivery_Outcome"]
    .value_counts(dropna=False)
    .to_frame("Records")
)

Diagnostic dataset shape: (180519, 59)

Valid duration records:
180519

Delivery outcome distribution:


,Records
Diagnostic_Delivery_Outcome,
Delayed,103400
On Schedule or Early,77119


## Validate Delivery Status Against Shipping Durations

In [6]:
status = (
    diagnostic_df["Delivery Status"]
    .astype("string")
    .str.strip()
    .str.lower()
)

recognized_statuses = {
    "late delivery": "Delayed",
    "advance shipping": "On Schedule or Early",
    "shipping on time": "On Schedule or Early",
    "on time": "On Schedule or Early",
    "on-time": "On Schedule or Early",
    "early": "On Schedule or Early",
    "delivered early": "On Schedule or Early"
}

diagnostic_df["Normalized_Delivery_Status"] = status

diagnostic_df["Status_Expected_Outcome"] = (
    status.map(recognized_statuses)
    .fillna("Unknown")
)

comparison_mask = (
    diagnostic_df["Valid_Duration_Record"]
    & diagnostic_df["Status_Expected_Outcome"].ne("Unknown")
)

diagnostic_df["Status_Duration_Consistency"] = "Not Comparable"

diagnostic_df.loc[
    comparison_mask,
    "Status_Duration_Consistency"
] = np.where(
    diagnostic_df.loc[
        comparison_mask, "Diagnostic_Delivery_Outcome"
    ].eq(
        diagnostic_df.loc[
            comparison_mask, "Status_Expected_Outcome"
        ]
    ),
    "Consistent",
    "Mismatch"
)

consistency_summary = (
    diagnostic_df["Status_Duration_Consistency"]
    .value_counts(dropna=False)
    .rename_axis("Consistency_Status")
    .reset_index(name="Record_Count")
)

print("Delivery Status Consistency Summary:")
display(consistency_summary)

# Percentage of comparable records with mismatches.
comparable_count = int(comparison_mask.sum())

mismatch_count = int(
    diagnostic_df["Status_Duration_Consistency"]
    .eq("Mismatch")
    .sum()
)

mismatch_rate = (
    mismatch_count / comparable_count * 100
    if comparable_count > 0
    else np.nan
)

print("\nComparable Records:", comparable_count)
print("Mismatch Records:", mismatch_count)
print(f"Mismatch Rate: {mismatch_rate:.2f}%")

Delivery Status Consistency Summary:


,Consistency_Status,Record_Count
0,Consistent,172765
1,Not Comparable,7754



Comparable Records: 172765
Mismatch Records: 0
Mismatch Rate: 0.00%


## Shipping-Mode Efficiency Diagnostics

In [7]:
valid_df = diagnostic_df.loc[
    diagnostic_df["Valid_Duration_Record"]
].copy()

shipping_mode_diagnostics = (
    valid_df.groupby("Shipping Mode", dropna=False)
    .agg(
        Shipment_Records=("Shipping Mode", "size"),
        Average_Actual_Days=("Actual_Shipping_Days", "mean"),
        Average_Scheduled_Days=("Scheduled_Shipping_Days", "mean"),
        Average_Delay_Days=("Diagnostic_Delay_Days", "mean"),
        Delayed_Records=(
            "Diagnostic_Delivery_Outcome",
            lambda x: x.eq("Delayed").sum()
        )
    )
    .reset_index()
)

shipping_mode_diagnostics["Delay_Rate_Percent"] = (
    shipping_mode_diagnostics["Delayed_Records"]
    / shipping_mode_diagnostics["Shipment_Records"]
    * 100
)

shipping_mode_diagnostics = (
    shipping_mode_diagnostics
    .sort_values("Delay_Rate_Percent", ascending=False)
)

display(shipping_mode_diagnostics.round(2))

,Shipping Mode,Shipment_Records,Average_Actual_Days,Average_Scheduled_Days,Average_Delay_Days,Delayed_Records,Delay_Rate_Percent
0,First Class,27814,2.00,1.0,1.00,27814,100.00
2,Second Class,35216,3.99,2.0,1.99,28078,79.73
1,Same Day,9737,0.48,0.0,0.48,4657,47.83
3,Standard Class,107752,4.00,4.0,-0.00,42851,39.77


## Regional Performance Diagnostics

In [8]:
regional_diagnostics = (
    valid_df.groupby("Order Region", dropna=False)
    .agg(
        Shipment_Records=("Order Region", "size"),
        Average_Actual_Days=("Actual_Shipping_Days", "mean"),
        Average_Scheduled_Days=("Scheduled_Shipping_Days", "mean"),
        Average_Delay_Days=("Diagnostic_Delay_Days", "mean"),
        Delayed_Records=(
            "Diagnostic_Delivery_Outcome",
            lambda x: x.eq("Delayed").sum()
        )
    )
    .reset_index()
)

regional_diagnostics["Delay_Rate_Percent"] = (
    regional_diagnostics["Delayed_Records"]
    / regional_diagnostics["Shipment_Records"]
    * 100
)

regional_diagnostics = regional_diagnostics.sort_values(
    "Delay_Rate_Percent",
    ascending=False
)

display(regional_diagnostics.round(2))

,Order Region,Shipment_Records,Average_Actual_Days,Average_Scheduled_Days,Average_Delay_Days,Delayed_Records,Delay_Rate_Percent
2,Central Africa,1677,3.56,2.92,0.64,1018,60.70
22,Western Europe,27109,3.50,2.90,0.60,15863,58.52
13,South Asia,7731,3.50,2.90,0.60,4523,58.50
14,South of USA,4045,3.49,2.91,0.58,2350,58.10
15,Southeast Asia,9539,3.50,2.94,0.56,5531,57.98
6,East of USA,6915,3.50,2.92,0.58,4009,57.98
20,West Asia,6009,3.49,2.92,0.57,3455,57.50
5,East Africa,1852,3.51,2.94,0.57,1064,57.45
8,Eastern Europe,3920,3.50,2.92,0.58,2252,57.45
3,Central America,28341,3.51,2.95,0.56,16224,57.25


## Market-Level Diagnostics

In [9]:
market_diagnostics = (
    valid_df.groupby("Market", dropna=False)
    .agg(
        Shipment_Records=("Market", "size"),
        Average_Actual_Days=("Actual_Shipping_Days", "mean"),
        Average_Scheduled_Days=("Scheduled_Shipping_Days", "mean"),
        Average_Delay_Days=("Diagnostic_Delay_Days", "mean"),
        Delayed_Records=(
            "Diagnostic_Delivery_Outcome",
            lambda x: x.eq("Delayed").sum()
        )
    )
    .reset_index()
)

market_diagnostics["Delay_Rate_Percent"] = (
    market_diagnostics["Delayed_Records"]
    / market_diagnostics["Shipment_Records"]
    * 100
)

market_diagnostics = market_diagnostics.sort_values(
    "Delay_Rate_Percent",
    ascending=False
)

display(market_diagnostics.round(2))

,Market,Shipment_Records,Average_Actual_Days,Average_Scheduled_Days,Average_Delay_Days,Delayed_Records,Delay_Rate_Percent
1,Europe,50252,3.49,2.92,0.57,28989,57.69
3,Pacific Asia,41260,3.50,2.93,0.57,23649,57.32
4,USCA,25799,3.48,2.92,0.57,14744,57.15
2,LATAM,51594,3.51,2.95,0.56,29420,57.02
0,Africa,11614,3.51,2.95,0.56,6598,56.81


## Creating General Data Quantity Diagnostic Reports

In [10]:
quality_columns = [
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Delivery Status",
    "Late_delivery_risk",
    "Shipping Mode",
    "Order Region",
    "Market"
]

quality_records = []

for column in quality_columns:

    if column not in df.columns:
        continue

    series = df[column]

    quality_records.append({
        "Column": column,
        "Data_Type": str(series.dtype),
        "Total_Records": len(series),
        "Missing_Records": int(series.isna().sum()),
        "Missing_Percent": round(
            series.isna().mean() * 100, 2
        ),
        "Unique_Non_Missing_Values": int(
            series.nunique(dropna=True)
        )
    })

data_quality_diagnostics = pd.DataFrame(quality_records)

display(data_quality_diagnostics)

,Column,Data_Type,Total_Records,Missing_Records,Missing_Percent,Unique_Non_Missing_Values
0,Days for shipping (real),int64,180519,0,0.0,7
1,Days for shipment (scheduled),int64,180519,0,0.0,4
2,Delivery Status,str,180519,0,0.0,4
3,Late_delivery_risk,int64,180519,0,0.0,2
4,Shipping Mode,str,180519,0,0.0,4
5,Order Region,str,180519,0,0.0,23
6,Market,str,180519,0,0.0,5


### checking for invalide shipping duartions

In [11]:
duration_validation = pd.DataFrame([
    {
        "Check": "Missing actual shipping duration",
        "Record_Count": int(
            diagnostic_df["Actual_Shipping_Days"].isna().sum()
        )
    },
    {
        "Check": "Missing scheduled shipping duration",
        "Record_Count": int(
            diagnostic_df["Scheduled_Shipping_Days"].isna().sum()
        )
    },
    {
        "Check": "Negative actual shipping duration",
        "Record_Count": int(
            diagnostic_df["Actual_Shipping_Days"].lt(0).sum()
        )
    },
    {
        "Check": "Negative scheduled shipping duration",
        "Record_Count": int(
            diagnostic_df["Scheduled_Shipping_Days"].lt(0).sum()
        )
    },
    {
        "Check": "Zero-day actual shipping duration",
        "Record_Count": int(
            diagnostic_df["Actual_Shipping_Days"].eq(0).sum()
        )
    },
    {
        "Check": "Valid duration pairs",
        "Record_Count": int(
            diagnostic_df["Valid_Duration_Record"].sum()
        )
    }
])

display(duration_validation)

,Check,Record_Count
0,Missing actual shipping duration,0
1,Missing scheduled shipping duration,0
2,Negative actual shipping duration,0
3,Negative scheduled shipping duration,0
4,Zero-day actual shipping duration,5080
5,Valid duration pairs,180519


## Consolidated Logistics Diagnostic Summary

In [12]:
diagnostic_summary = pd.DataFrame([
    {
        "Diagnostic_Area": "Data Quality",
        "Metric": "Total Dataset Records",
        "Value": len(df),
        "Unit": "Records"
    },
    {
        "Diagnostic_Area": "Data Quality",
        "Metric": "Duplicate Records",
        "Value": int(df.duplicated().sum()),
        "Unit": "Records"
    },
    {
        "Diagnostic_Area": "Data Quality",
        "Metric": "Missing Cells",
        "Value": int(df.isna().sum().sum()),
        "Unit": "Cells"
    },
    {
        "Diagnostic_Area": "Shipping Efficiency",
        "Metric": "Valid Duration Records",
        "Value": int(
            diagnostic_df["Valid_Duration_Record"].sum()
        ),
        "Unit": "Records"
    },
    {
        "Diagnostic_Area": "Delivery Consistency",
        "Metric": "Comparable Status-Duration Records",
        "Value": comparable_count,
        "Unit": "Records"
    },
    {
        "Diagnostic_Area": "Delivery Consistency",
        "Metric": "Status-Duration Mismatches",
        "Value": mismatch_count,
        "Unit": "Records"
    },
    {
        "Diagnostic_Area": "Delivery Consistency",
        "Metric": "Status-Duration Mismatch Rate",
        "Value": mismatch_rate,
        "Unit": "Percent"
    }
])

display(diagnostic_summary)

,Diagnostic_Area,Metric,Value,Unit
0,Data Quality,Total Dataset Records,180519.0,Records
1,Data Quality,Duplicate Records,0.0,Records
2,Data Quality,Missing Cells,5080.0,Cells
3,Shipping Efficiency,Valid Duration Records,180519.0,Records
4,Delivery Consistency,Comparable Status-Duration Records,172765.0,Records
5,Delivery Consistency,Status-Duration Mismatches,0.0,Records
6,Delivery Consistency,Status-Duration Mismatch Rate,0.0,Percent


## Export Outputs

In [13]:
output_files = {
    "APL_Day19_Data_Quality_Diagnostics.csv":
        data_quality_diagnostics,

    "APL_Day19_Duration_Validation.csv":
        duration_validation,

    "APL_Day19_Status_Consistency_Summary.csv":
        consistency_summary,

    "APL_Day19_Shipping_Mode_Diagnostics.csv":
        shipping_mode_diagnostics,

    "APL_Day19_Regional_Diagnostics.csv":
        regional_diagnostics,

    "APL_Day19_Market_Diagnostics.csv":
        market_diagnostics,

    "APL_Day19_Consolidated_Diagnostic_Summary.csv":
        diagnostic_summary
}

for filename, output_df in output_files.items():

    output_path = ANALYSIS_DATA_DIR / filename

    output_df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"Saved: {output_path} | "
        f"Rows: {len(output_df)} | "
        f"Columns: {len(output_df.columns)}"
    )

print("\nDay 19 diagnostic exports completed.")

Saved: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\data\analysis_data\APL_Day19_Data_Quality_Diagnostics.csv | Rows: 7 | Columns: 6
Saved: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\data\analysis_data\APL_Day19_Duration_Validation.csv | Rows: 6 | Columns: 2
Saved: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\data\analysis_data\APL_Day19_Status_Consistency_Summary.csv | Rows: 2 | Columns: 2
Saved: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\data\analysis_data\APL_Day19_Shipping_Mode_Diagnostics.csv | Rows: 4 | Columns: 7
Saved: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\dat

## Validation

In [15]:
validation_results = []

for filename, original_df in output_files.items():

    output_path = ANALYSIS_DATA_DIR / filename

    file_exists = output_path.exists()

    validation_results.append({
        "File_Name": filename,
        "Exists": file_exists,
        "Expected_Rows": len(original_df),
        "Expected_Columns": len(original_df.columns),
        "File_Size_Bytes": (
            output_path.stat().st_size
            if file_exists else 0
        )
    })

output_validation = pd.DataFrame(validation_results)

display(output_validation)

assert output_validation["Exists"].all(), (
    "One or more Day 19 output files are missing."
)

assert (
    output_validation["File_Size_Bytes"] > 0
).all(), (
    "One or more Day 19 output files are empty."
)

print("ALL DAY 19 OUTPUT FILES VALIDATED.")


,File_Name,Exists,Expected_Rows,Expected_Columns,File_Size_Bytes
0,APL_Day19_Data_Quality_Diagnostics.csv,True,7,6,364
1,APL_Day19_Duration_Validation.csv,True,6,2,244
2,APL_Day19_Status_Consistency_Summary.csv,True,2,2,76
3,APL_Day19_Shipping_Mode_Diagnostics.csv,True,4,7,432
4,APL_Day19_Regional_Diagnostics.csv,True,23,7,2377
5,APL_Day19_Market_Diagnostics.csv,True,5,7,599
6,APL_Day19_Consolidated_Diagnostic_Summary.csv,True,7,4,436


ALL DAY 19 OUTPUT FILES VALIDATED.
